In [3]:
##### Calculate the spearman rank coefficient between predicted capital/labor and other existing datasets

import os
import rioxarray as rio
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from rasterio.enums import Resampling

In [6]:
##### Load data

# Get the current working directory
cd = os.path.dirname(os.getcwd())

# Import data
capital_intensity = rio.open_rasterio(f"{cd}/Results/Raster_model/capital_intensity_USD_per_tonne_reprojected.tif")
labor = rio.open_rasterio(f"{cd}/Results/Raster_model/reprojected/rescaled_jobs.tif")

capital_validation = rio.open_rasterio(f"{cd}/Data/Raw/Roman_mechanization/Agricultural Mechanization_reprojected.tif")
labor_validation = rio.open_rasterio(f"{cd}/Data/Raw/Zulueta_ag_workforce/Baseline/baseline_2020_corrected_ag_pop_reprojected.tif")

In [8]:
### Calculate capital metrics

def align_and_flatten(*rasters):
    ref = rasters[0]
    aligned = [ref] + [r.rio.reproject_match(ref) for r in rasters[1:]]
    arrays = [a.values.squeeze().ravel() for a in aligned]
    valid = np.ones_like(arrays[0], dtype=bool)
    for arr in arrays:
        valid &= np.isfinite(arr)
    return [arr[valid] for arr in arrays], valid.sum()

(val_flat, est_flat), n = align_and_flatten(
    capital_validation, capital_intensity
)

# Spearman rank coefficient
rho, pval = stats.spearmanr(val_flat, est_flat)

# Print
results = pd.Series({
    "N_pixels": n,
    "Spearman_rho": rho,
    "Spearman_pval": pval,
})

results_rounded = results.round(3)
print(results_rounded)

N_pixels         518558.000
Spearman_rho          0.262
Spearman_pval         0.000
dtype: float64


In [12]:
### Calculate labor metrics

def align_and_flatten(*rasters):
    ref = rasters[0]
    aligned = [ref] + [r.rio.reproject_match(ref) for r in rasters[1:]]
    arrays = [a.values.squeeze().ravel() for a in aligned]
    valid = np.ones_like(arrays[0], dtype=bool)
    for arr in arrays:
        valid &= np.isfinite(arr)
    return [arr[valid] for arr in arrays], valid.sum()

(val_flat, est_flat), n = align_and_flatten(
    labor_validation, labor
)

# R2 (actual scale and log scale)
r2 = r2_score(val_flat, est_flat)

eps = 1
log_val = np.log(val_flat + eps)
log_est = np.log(est_flat + eps)
log_r2 = r2_score(log_val, log_est)

# Spearman rank coefficient
rho, pval = stats.spearmanr(val_flat, est_flat)

# RSME, MAE, bias, pbias
rmse = np.sqrt(mean_squared_error(val_flat, est_flat))
log_rmse = np.sqrt(mean_squared_error(log_val, log_est))

mae = mean_absolute_error(val_flat, est_flat)
bias = (est_flat - val_flat).mean()
pct_bias = bias / val_flat.mean() * 100

# Print
results = pd.Series({
    "N_pixels": n,
    "R2": r2,
    "Log_R2": log_r2,
    "Spearman_rho": rho,
    "Spearman_pval": pval,
    "RMSE": rmse,
    "Log_RMSE": log_rmse,
    "MAE": mae,
    "Mean_bias": bias,
    "Pct_bias": pct_bias
})

results_rounded = results.round(3)
print(results_rounded)

N_pixels         1078034.000
R2                     0.482
Log_R2                 0.738
Spearman_rho           0.885
Spearman_pval          0.000
RMSE                2245.473
Log_RMSE               1.287
MAE                  719.336
Mean_bias           -390.572
Pct_bias             -33.378
dtype: float64


In [ ]:
# N_pixels (1,078,034): The number of pixels compared between your model and the validation data.
# R² (0.482): Your model explains about 48% of the variation in actual worker counts on a raw-number scale.
# Log_R² (0.738): When you look at proportional (not raw) differences, your model explains about 74% of the variation — a much better fit.
# Spearman_rho (0.885): Your model is very good at correctly ranking areas from low to high labor, even if it's off on exact numbers.
# Spearman_pval (0.000): That ranking relationship is statistically real, not due to chance.
# RMSE (2245.5): On average, factoring in a few big misses, your estimates are off by about 2,245 workers per pixel.
# Log_RMSE (1.287): On a proportional basis, your typical estimate is off by roughly a factor of ~3.6x in either direction.
# MAE (719.3): Ignoring the extreme outliers, your typical pixel-level estimate is off by about 719 workers.
# Mean_bias (-390.6): On average, your model underestimates the number of workers by about 391 per pixel.
# Pct_bias (-33.4%): Overall, your model estimates about a third fewer workers than the validation data shows.